In [1]:
import numpy as np
import pandas as pd

import time
import warnings

import json
import os

import optuna
from sklearn.preprocessing import StandardScaler

from quantile_forest import RandomForestQuantileRegressor
import random
random.seed(42)
np.random.seed(42)
warnings.filterwarnings('ignore')

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Fujian data
df = pd.read_csv('res_new.csv')

a = np.array(df.values)

def split_data(data, horizon, window):
    n, m = data.shape
    X = np.zeros((n - window - horizon, window))
    Y = np.zeros((n - window - horizon,1))
    EX = np.zeros((n - window - horizon, 2))

    for i in range(n - window - horizon):
        start = i
        end = start + window

        X[i, :] = data[start:end,0]
        Y[i] = data[end + horizon - 1,0]
        EX[i,:] = data[end + horizon - 1,1:2]

    return X, Y, EX

X,Y,EX = split_data(a,1,96)
X_arg = np.hstack((X,EX))
n1=X_arg.shape[0]

train_X, train_Y = X_arg[:round(0.8 * n1)], Y[:round(0.8 * n1)]
val_X, val_Y = X_arg[round(0.8 * n1):round(0.9 * n1)], Y[round(0.8 * n1):round(0.9 * n1)]
test_X, test_Y = X_arg[round(0.9 * n1):], Y[round(0.9 * n1):]

scaler_X = StandardScaler()
x_train = scaler_X.fit_transform(train_X)
x_val = scaler_X.transform(val_X)
x_test = scaler_X.transform(test_X)

scaler_Y = StandardScaler()
y_train = scaler_Y.fit_transform(train_Y)
y_val = scaler_Y.transform(val_Y)
y_test = scaler_Y.transform(test_Y)

tau_values = [round(i, 2) for i in np.arange(0.05, 1.0, 0.05)]

In [3]:
def tilted_absolute_loss(y_true, y_pred, quantiles):
    total_loss = 0.0
    for i, tau in enumerate(quantiles):
        residual = y_true - y_pred[:, i]
        loss = np.maximum(tau * residual, (tau - 1) * residual)
        total_loss += np.sum(loss)
    
    return total_loss / (len(y_true) * len(quantiles))

Optuna searching (Option C)

In [12]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 10, 300)

    start_time = time.time()

    model = RandomForestQuantileRegressor(
        n_estimators=n_estimators,
        n_jobs=-1,
        random_state=42)

    model.fit(x_train, y_train.flatten())

    y_pred_val_scaled = model.predict(x_val, quantiles=tau_values)

    y_pred_val_original = scaler_Y.inverse_transform(y_pred_val_scaled)

    current_loss = tilted_absolute_loss(val_Y, y_pred_val_original, tau_values)

    trial_time = time.time() - start_time
    trial.set_user_attr('time', trial_time)

    if trial.number % 5 == 0:
        print(f"Trial {trial.number}: n_est={n_estimators}, loss={current_loss:.6f}, time={trial_time:.2f}s")

    return current_loss

In [ ]:
def optimize_hyperparams(n_trials=100):
    """
    Optimize Random Forest hyperparameters using Optuna.
    """
    print(f"\nStarting Optuna optimization, {n_trials} trials...")
    print("Searching hyperparameter: n_estimators")

    # Create Optuna study
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    # Run optimization
    start_time = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    total_time = time.time() - start_time

    print(f"\n{'='*60}")
    print(f"Optimization complete!")
    print(f"Total time: {total_time:.3f}s")

    # Display best parameters
    best_params = study.best_params
    print(f"Best hyperparameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    print(f"Best validation loss: {study.best_value:.6f}")

    # Display best trial details
    best_trial = study.best_trial
    print(f"\nBest trial details:")
    print(f"  Trial number: {best_trial.number}")
    print(f"  Validation loss: {best_trial.value:.6f}")
    print(f"  Runtime: {best_trial.user_attrs.get('time', 0):.2f}s")

    # Save optimization results
    results = {study.best_params}

    with open("best_paramsqrffj.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    return results

#Run for finding hyperparameters
optuna_results = optimize_hyperparams(n_trials=100)
params = optuna_results['n_estimators']
n_estimators= params['n_estimators']

Using hyperparameters directly, without optuna (Option B)

In [1]:
RESULTS_JSON = 'best_paramsqrffj.json'

with open(RESULTS_JSON, 'r', encoding='utf-8') as f:
    saved = json.load(f)

params = saved['n_estimators']
n_estimators= params['n_estimators']

Test set evaluation

In [5]:
def evaluate_on_test_set(n_estimators, save_predictions=True):
    """
    Evaluate model on the test set using given hyperparameters.
    """
    print(f"\nEvaluating on test set...")

    # Create model
    model = RandomForestQuantileRegressor(
        n_estimators=n_estimators,
        n_jobs=-1,
        random_state=42
    )

    # Train model (on standardized training data)
    print("Training model...")
    model.fit(x_train, y_train.flatten())

    # Predict on test set (in standardized scale)
    print("Predicting on test set...")
    y_pred_test_scaled = model.predict(x_test, quantiles=tau_values)

    # Inverse-transform predictions to original scale
    y_pred_test_original = scaler_Y.inverse_transform(y_pred_test_scaled)

    test_loss = tilted_absolute_loss(test_Y, y_pred_test_original, tau_values)

    print(f"\n{'='*60}")
    print(f"Test set evaluation results:")
    print(f"Test set mean pinball loss: {test_loss:.6f}")

    return test_loss, y_pred_test_original, model

In [ ]:
test_loss, test_predictions_matrix, final_model = evaluate_on_test_set(n_estimators)


Evaluating on test set...
Training model...


In [15]:
pred = pd.DataFrame(test_predictions_matrix)
pred.to_excel('predqrffj.xlsx', index=False)